# Modelltraining mit offiziellen Datensplits

Dieses Notebook verwendet die bereits im Quelldatensatz festgelegten Trainings-, Validierungs- und Testdaten. Die Modellwahl erfolgt per Cross-Validation ausschließlich auf dem Trainingssplit. Der Entscheidungsschwellenwert wird auf dem Validierungssplit bestimmt; der Testsplit bleibt bis zur finalen Bewertung unangetastet.

In [1]:
%pip install -q pandas scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    precision_recall_curve,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.constants import Column, MODEL_FEATURE_COLUMNS, OUTPUT_DIR

In [3]:
final_train_path = OUTPUT_DIR / "train" / "final.csv"
final_train = pd.read_csv(final_train_path)

required_columns = [
    Column.ID,
    Column.LABEL,
    Column.SPLIT,
    *MODEL_FEATURE_COLUMNS,
]
missing_columns = [
    column for column in required_columns
    if column not in final_train.columns
]
if missing_columns:
    raise ValueError(f"Fehlende Spalten in {final_train_path}: {missing_columns}")

if final_train[required_columns].isna().any().any():
    columns_with_nan = final_train[required_columns].columns[
        final_train[required_columns].isna().any()
    ].tolist()
    raise ValueError(f"Fehlende Featurewerte in: {columns_with_nan}")

if not final_train[Column.ID].is_unique:
    raise ValueError("Die IDs in final_train sind nicht eindeutig.")

expected_splits = {"train", "validation", "test"}
actual_splits = set(final_train[Column.SPLIT].unique())
if actual_splits != expected_splits:
    raise ValueError(
        f"Erwartete Splits: {sorted(expected_splits)}; "
        f"vorhanden: {sorted(actual_splits)}."
    )

for split_name in ["train", "validation", "test"]:
    split_labels = final_train.loc[
        final_train[Column.SPLIT] == split_name,
        Column.LABEL,
    ]
    class_counts = split_labels.value_counts().sort_index()
    if len(class_counts) != 2:
        raise ValueError(
            f"Split '{split_name}' enthält nicht beide Klassen: "
            f"{class_counts.to_dict()}"
        )

print(f"Datensatz: {final_train_path}")
print(f"Reviews: {len(final_train)}")
print("Verteilung nach Split und Label:")
print(
    pd.crosstab(
        final_train[Column.SPLIT],
        final_train[Column.LABEL],
    )
)

ValueError: Fehlende Spalten in /Users/malik/Documents/projects/fake-review-detection/data/output/train/final.csv: ['split', 'category_specificity']

In [ ]:
split_frames = {
    split_name: final_train[
        final_train[Column.SPLIT] == split_name
    ].copy()
    for split_name in ["train", "validation", "test"]
}

X_train = split_frames["train"][MODEL_FEATURE_COLUMNS]
y_train = split_frames["train"][Column.LABEL]
X_validation = split_frames["validation"][MODEL_FEATURE_COLUMNS]
y_validation = split_frames["validation"][Column.LABEL]
X_test = split_frames["test"][MODEL_FEATURE_COLUMNS]
y_test = split_frames["test"][Column.LABEL]

print(
    f"Training: {len(X_train)} | "
    f"Validierung: {len(X_validation)} | "
    f"Test: {len(X_test)}"
)
print("\nFake-Anteil:")
print(f"Training: {y_train.mean():.2%}")
print(f"Validierung: {y_validation.mean():.2%}")
print(f"Test: {y_test.mean():.2%}")

In [ ]:
models = {
    "Logistische Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=1_000,
            random_state=42,
            class_weight="balanced",
        ),
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=1,
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
}

cv_rows = []
for model_name, candidate in models.items():
    scores = cross_validate(
        candidate,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
    )
    row = {"model": model_name}
    for metric in scoring:
        values = scores[f"test_{metric}"]
        row[f"{metric}_mean"] = values.mean()
        row[f"{metric}_std"] = values.std()
    cv_rows.append(row)

cv_results = (
    pd.DataFrame(cv_rows)
    .set_index("model")
    .sort_values("f1_mean", ascending=False)
)
cv_summary_columns = [
    "accuracy_mean",
    "balanced_accuracy_mean",
    "precision_mean",
    "recall_mean",
    "f1_mean",
    "roc_auc_mean",
    "average_precision_mean",
]
print("5-fache Cross-Validation auf den Trainingsdaten:")
print(cv_results[cv_summary_columns].round(4).to_string())

best_model_name = cv_results.index[0]
model = models[best_model_name]
model.fit(X_train, y_train)
print(f"Ausgewähltes Modell: {best_model_name}")

In [ ]:
def select_f1_threshold(y_true, probability):
    precision, recall, thresholds = precision_recall_curve(
        y_true,
        probability,
    )
    precision_values = pd.Series(precision[:-1])
    recall_values = pd.Series(recall[:-1])
    denominator = precision_values + recall_values
    f1_scores = (
        2 * precision_values * recall_values
        / denominator.mask(denominator == 0)
    ).fillna(0).to_numpy()
    best_position = f1_scores.argmax()
    return thresholds[best_position]


def evaluate(model, X, y, dataset_name, threshold):
    probability = model.predict_proba(X)[:, 1]
    prediction = (probability >= threshold).astype(int)
    metrics = {
        "accuracy": accuracy_score(y, prediction),
        "balanced_accuracy": balanced_accuracy_score(y, prediction),
        "precision": precision_score(y, prediction, zero_division=0),
        "recall": recall_score(y, prediction, zero_division=0),
        "f1": f1_score(y, prediction, zero_division=0),
        "roc_auc": roc_auc_score(y, probability),
        "average_precision": average_precision_score(y, probability),
    }
    print(f"\n{dataset_name}")
    print(pd.Series(metrics).round(4))
    print("Konfusionsmatrix:")
    print(confusion_matrix(y, prediction))
    return metrics

validation_probability = model.predict_proba(X_validation)[:, 1]
decision_threshold = select_f1_threshold(
    y_validation,
    validation_probability,
)
print(f"Auf der Validierung gewählter Schwellenwert: {decision_threshold:.4f}")

validation_metrics = evaluate(
    model,
    X_validation,
    y_validation,
    "Validierung",
    decision_threshold,
)

test_metrics = evaluate(
    model,
    X_test,
    y_test,
    "Test",
    decision_threshold,
)

pd.DataFrame(
    [validation_metrics, test_metrics],
    index=["validation", "test"],
).round(4)